<a href="https://colab.research.google.com/github/peperjet/deep-learning/blob/main/KoBERT_260513.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

KoBERT 기계 독해

1. 전처리부터 시작

In [ ]:
# KoBERT Transformers (Hugging Face 호환 버전)
!pip install transformers==4.10.0
!pip install sentencepiece
!pip install git+https://github.com/monologg/KoBERT-Transformers.git


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.6/51.6 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.7/212.7 kB 9.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 68.2 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)
  Cloning https://github.com/monologg/KoBERT-Transformers.git to /tmp/pip-req-build-lxpbdxtd
  Running command git clon

In [ ]:
# KorQuAD 1.0 다운로드
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/korquad/korquad.github.io/master/dataset/KorQuAD_v1.0_train.json",
    filename="KorQuAD_v1.0_train.json"
)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/korquad/korquad.github.io/master/dataset/KorQuAD_v1.0_dev.json",
    filename="KorQuAD_v1.0_dev.json"
)

('KorQuAD_v1.0_dev.json', <http.client.HTTPMessage at 0x7a14a8f6f680>)

2. 데이터 구조 확인

In [ ]:
import json

with open("KorQuAD_v1.0_train.json", "r") as f:
    train_data = json.load(f)

# 구조 파악
sample = train_data['data'][0]['paragraphs'][0]
print("본문:", sample['context'][:100])
print("질문:", sample['qas'][0]['question'])
print("정답:", sample['qas'][0]['answers'][0]['text'])
print("정답 시작 위치:", sample['qas'][0]['answers'][0]['answer_start'])

본문: 1839년 바그너는 괴테의 파우스트을 처음 읽고 그 내용에 마음이 끌려 이를 소재로 해서 하나의 교향곡을 쓰려는 뜻을 갖는다. 이 시기 바그너는 1838년에 빛 독촉으로 산전수전을
질문: 바그너는 괴테의 파우스트를 읽고 무엇을 쓰고자 했는가?
정답: 교향곡
정답 시작 위치: 54


3. 전처리 함수

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from kobert_transformers import get_tokenizer

tokenizer = get_tokenizer()

def get_dataset(file_path, max_seq_length=512):
    with open(file_path, 'r') as f:
        data = json.load(f)

    input_ids_list, attention_mask_list = [], []
    token_type_ids_list = [], []
    start_positions, end_positions = [], []

    for article in data['data']:
        for para in article['paragraphs']:
            context = para['context']
            for qa in para['qas']:
                question = qa['question']
                answer_text = qa['answers'][0]['text']
                answer_start_char = qa['answers'][0]['answer_start']

                # 토크나이징
                encoding = tokenizer(
                    question,
                    context,
                    max_length=max_seq_length,
                    truncation=True,
                    padding='max_length',
                    return_offsets_mapping=True,
                    return_tensors='pt'
                )

                # 정답 토큰 위치 찾기
                offset_mapping = encoding['offset_mapping'][0]
                answer_end_char = answer_start_char + len(answer_text)

                start_pos, end_pos = 0, 0
                for idx, (start, end) in enumerate(offset_mapping):
                    if start <= answer_start_char < end:
                        start_pos = idx
                    if start < answer_end_char <= end:
                        end_pos = idx
                        break

                input_ids_list.append(encoding['input_ids'][0])
                attention_mask_list.append(encoding['attention_mask'][0])
                token_type_ids_list.append(encoding['token_type_ids'][0])
                start_positions.append(torch.tensor(start_pos))
                end_positions.append(torch.tensor(end_pos))

    return (input_ids_list, attention_mask_list,
            token_type_ids_list, start_positions, end_positions)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/263 [00:00<?, ?B/s]

tokenizer_78b3253a26.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [ ]:
# 토크나이저 잘 불러와졌는지 확인
sample = tokenizer("안녕하세요", "오늘 날씨가 좋네요")
print(sample)

{'input_ids': [2, 3135, 5724, 7814, 3, 3419, 1408, 5330, 4204, 5703, 3], 'token_type_ids': [0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


input_ids — 단어를 숫자로 변환

token_type_ids — 어느 문장인지 구분

attention_mask — 진짜 토큰 vs 패딩

In [ ]:
# KorQuAD 데이터 다운로드
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/korquad/korquad.github.io/master/dataset/KorQuAD_v1.0_train.json",
    filename="KorQuAD_v1.0_train.json"
)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/korquad/korquad.github.io/master/dataset/KorQuAD_v1.0_dev.json",
    filename="KorQuAD_v1.0_dev.json"
)
print("다운로드 완료!")

다운로드 완료!


In [ ]:
import json

with open("KorQuAD_v1.0_train.json", "r") as f:
    train_data = json.load(f)

# 샘플 하나만 꺼내보기
sample = train_data['data'][0]['paragraphs'][0]
print("본문:", sample['context'][:100])
print()
print("질문:", sample['qas'][0]['question'])
print()
print("정답:", sample['qas'][0]['answers'][0]['text'])
print("정답 시작 위치:", sample['qas'][0]['answers'][0]['answer_start'])

본문: 1839년 바그너는 괴테의 파우스트을 처음 읽고 그 내용에 마음이 끌려 이를 소재로 해서 하나의 교향곡을 쓰려는 뜻을 갖는다. 이 시기 바그너는 1838년에 빛 독촉으로 산전수전을

질문: 바그너는 괴테의 파우스트를 읽고 무엇을 쓰고자 했는가?

정답: 교향곡
정답 시작 위치: 54


In [ ]:
# 글자 위치 vs 토큰 위치 확인
context = sample['context']
question = sample['qas'][0]['question']
answer_start_char = sample['qas'][0]['answers'][0]['answer_start']
answer_text = sample['qas'][0]['answers'][0]['text']

print(f"정답 글자: '{answer_text}'")
print(f"글자 기준 시작: {answer_start_char}")
print(f"글자 기준 끝: {answer_start_char + len(answer_text)}")
print()

# offset_mapping으로 토큰 위치 확인
encoding = tokenizer(
    question,
    context,
    return_offsets_mapping=True,
    truncation=True,
    max_length=512
)
print("offset_mapping 앞부분:", encoding['offset_mapping'][:15])

정답 글자: '교향곡'
글자 기준 시작: 54
글자 기준 끝: 57



NotImplementedError: return_offset_mapping is not available when using Python tokenizers. To use this feature, change your tokenizer to one deriving from transformers.PreTrainedTokenizerFast. More information on available tokenizers at https://github.com/huggingface/transformers/pull/2674

In [ ]:
# offset_mapping 대신 직접 토큰 위치 찾기
encoding = tokenizer(
    question,
    context,
    truncation=True,
    max_length=512
)

# 정답 텍스트를 토큰으로 변환해서 위치 찾기
answer_tokens = tokenizer.encode(answer_text, add_special_tokens=False)
input_ids = encoding['input_ids']

print("input_ids:", input_ids[:20])
print("정답 토큰:", answer_tokens)

input_ids: [2, 2186, 5538, 5691, 5760, 1101, 7618, 7095, 4799, 7005, 6691, 6116, 3824, 5439, 2095, 6884, 7088, 3084, 5439, 7147]
정답 토큰: [1103, 7886, 5443]


In [ ]:
# 정답 토큰 위치 찾기 (offset_mapping 없이)
def find_answer_token_idx(input_ids, answer_tokens):
    for i in range(len(input_ids) - len(answer_tokens) + 1):
        if input_ids[i:i+len(answer_tokens)] == answer_tokens:
            start = i
            end = i + len(answer_tokens) - 1
            return start, end
    return 0, 0  # 못 찾으면 0, 0

start_pos, end_pos = find_answer_token_idx(input_ids, answer_tokens)
print(f"정답 토큰: {answer_tokens}")
print(f"start_position: {start_pos}")
print(f"end_position: {end_pos}")
print(f"검증: {tokenizer.decode(input_ids[start_pos:end_pos+1])}")

정답 토큰: [1103, 7886, 5443]
start_position: 56
end_position: 58
검증: 교향곡


In [ ]:
from torch.utils.data import Dataset
import torch

class KorQuADDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_seq_length=512):
        with open(file_path, 'r') as f:
            data = json.load(f)

        self.input_ids = []
        self.attention_masks = []
        self.token_type_ids = []
        self.start_positions = []
        self.end_positions = []

        for article in data['data']:
            for para in article['paragraphs']:
                context = para['context']
                for qa in para['qas']:
                    question = qa['question']
                    answer_text = qa['answers'][0]['text']

                    encoding = tokenizer(
                        question, context,
                        truncation=True,
                        max_length=max_seq_length,
                        padding='max_length'
                    )

                    answer_tokens = tokenizer.encode(
                        answer_text, add_special_tokens=False
                    )
                    input_ids = encoding['input_ids']
                    start, end = find_answer_token_idx(input_ids, answer_tokens)

                    self.input_ids.append(torch.tensor(input_ids))
                    self.attention_masks.append(torch.tensor(encoding['attention_mask']))
                    self.token_type_ids.append(torch.tensor(encoding['token_type_ids']))
                    self.start_positions.append(torch.tensor(start))
                    self.end_positions.append(torch.tensor(end))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_masks[idx],
            'token_type_ids': self.token_type_ids[idx],
            'start_positions': self.start_positions[idx],
            'end_positions': self.end_positions[idx]
        }

In [ ]:
print("데이터셋 만드는 중... 잠깐 기다려요")
dev_dataset = KorQuADDataset("KorQuAD_v1.0_dev.json", tokenizer)
print(f"완료! 총 {len(dev_dataset)}개")
print()

# 첫 번째 샘플 확인
sample = dev_dataset[0]
print("input_ids shape:", sample['input_ids'].shape)
print("start_position:", sample['start_positions'])
print("end_position:", sample['end_positions'])

데이터셋 만드는 중... 잠깐 기다려요


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

완료! 총 5774개

input_ids shape: torch.Size([512])
start_position: tensor(22)
end_position: tensor(28)


DataLoader 만들고 모델 불러오기

In [ ]:
from torch.utils.data import DataLoader
from transformers import BertForQuestionAnswering

# DataLoader
dev_loader = DataLoader(dev_dataset, batch_size=16, shuffle=False)

# 모델
model = BertForQuestionAnswering.from_pretrained('monologg/kobert')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print("device:", device)
print("모델 로드 완료!")

config.json:   0%|          | 0.00/426 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at monologg/kobert and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


device: cuda
모델 로드 완료!


학습 루프 짜기

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=3e-5)

print("학습 준비 완료!")

학습 준비 완료!


AdamW가 최신 transformers에서 제거되었으므로 PyTorch 꺼 사용함.

In [ ]:
# 학습루프
from tqdm import tqdm

EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in tqdm(dev_loader, desc=f"Epoch {epoch+1}"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        start_positions = batch['start_positions'].to(device)
        end_positions = batch['end_positions'].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            start_positions=start_positions,
            end_positions=end_positions
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dev_loader)
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f}")

Epoch 1: 100%|██████████| 361/361 [09:37<00:00,  1.60s/it]


Epoch 1 | Loss: 3.3132


Epoch 2: 100%|██████████| 361/361 [09:39<00:00,  1.61s/it]


Epoch 2 | Loss: 2.0302


Epoch 3: 100%|██████████| 361/361 [09:39<00:00,  1.60s/it]

Epoch 3 | Loss: 1.4946


질문 던져보기

In [ ]:
model.eval()

question = "바그너는 괴테의 파우스트를 읽고 무엇을 쓰고자 했는가?"
context = "1839년 바그너는 괴테의 파우스트을 처음 읽고 그 내용에 마음이 끌려 이를 소재로 해서 하나의 교향곡을 쓰려는 뜻을 갖는다."

encoding = tokenizer(
    question, context,
    truncation=True,
    max_length=512,
    return_tensors='pt'
).to(device)

with torch.no_grad():
    outputs = model(**encoding)

start_idx = torch.argmax(outputs.start_logits)
end_idx = torch.argmax(outputs.end_logits) + 1

answer = tokenizer.decode(encoding['input_ids'][0][start_idx:end_idx])
print("질문:", question)
print("정답:", answer)

질문: 바그너는 괴테의 파우스트를 읽고 무엇을 쓰고자 했는가?
정답: 교향곡


In [24]:
# 메타데이터 정리

!pip install nbformat

import nbformat

notebook_path = "/content/drive/MyDrive/Colab Notebooks/KoBERT_260513.ipynb"

with open(notebook_path, "r") as f:
    nb = nbformat.read(f, as_version=4)

# 노트북 레벨의 위젯 메타데이터 제거
if 'widgets' in nb.metadata:
    del nb.metadata['widgets']

# 각 셀의 위젯 메타데이터 제거
for cell in nb.cells:
    if 'metadata' in cell and 'widgets' in cell.metadata:
        del cell.metadata['widgets']

with open(notebook_path, "w") as f:
    nbformat.write(nb, f)

print("완료!")

완료!
